# Sentiment Analysis Using Word2Vec Embeddings and Logistic Regression

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

This project extends an earlier phase of the same experimentation lab, which used frequency-based representations (Bag-of-Words and TF-IDF) to establish classical baselines. In this phase, I move to **dense, distributed word representations** using **Word2Vec**, and investigate how context-aware embeddings compare to frequency-based features for sentiment classification.

The primary objective is not only to achieve strong predictive performance, but to systematically examine how embedding hyperparameters, sentence-vector aggregation strategy, and preprocessing choices interact — and to understand *why* certain configurations outperform others, rather than just reporting which one wins.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:
* Positive
* Negative

The same train/test split used in the BoW/TF-IDF phase is reused here, with Word2Vec trained only on the training partition to avoid data leakage.

---

## Why Word2Vec, and Why the Preprocessing Changes

Frequency-based methods like BoW and TF-IDF treat words as independent, count-based features — they have no notion of word meaning or context. Word2Vec instead learns dense vector representations from the **local context** each word appears in, capturing semantic relationships (e.g., "great" and "wonderful" ending up close in vector space).

This shift in representation motivates a deliberate shift in preprocessing:

* **Stopwords are retained** (rather than removed, as in the TF-IDF phase), since function words supply the local context Word2Vec learns from — removing them shrinks the effective context window and can degrade embedding quality.
* **No lemmatization** is applied, to preserve distinctions between word forms (e.g., "acting" vs "act") that may carry different contextual signal.
* **Negation handling** carries over from the earlier phase's preprocessing discipline, since negation is critical to sentiment polarity.

---

## Project Pipeline

### 1. Text Preprocessing
* Contraction expansion
* Lowercasing
* Possessive stripping and regex-based cleaning
* Stopwords retained (no removal), no lemmatization
* Corpus construction as tokenized word lists

### 2. Embedding Training
* Word2Vec (Gensim) trained on the training corpus only
* Skip-gram vs. CBOW comparison
* Hyperparameter tuning: `vector_size`, `window`, `epochs`, `min_count`
* Qualitative validation via nearest-neighbor inspection (e.g., words most similar to "great")

### 3. Feature Construction (Sentence Vectors)
* Unweighted mean-pooling of word vectors per review
* TF-IDF-weighted mean-pooling, as a comparison strategy
* Out-of-vocabulary and empty-token edge-case handling

### 4. Model Training
Classical classifiers suited to dense, continuous-valued features are evaluated, including:
* Logistic Regression
* Support Vector Machines (SVM)
* Random Forest
* XGBoost
* Artificial Neural Network (ANN)
* K-Nearest Neighbors
* Gaussian Naive Bayes

(Multinomial and Bernoulli Naive Bayes are excluded in this phase, as they assume non-negative, count-like input — an assumption Word2Vec's dense, signed vectors violate.)

### 5. Model Evaluation
Performance is assessed using the same metric set as the earlier phase, for direct comparability:
* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score (computed from predicted probabilities, not hard labels)
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single embedding and calling it final, this notebook follows the same experimentation-driven methodology as the BoW/TF-IDF phase. Word2Vec's hyperparameters were tuned incrementally — one dimension at a time — across:

* `sg` (skip-gram vs. CBOW)
* `window` size
* `vector_size`
* `epochs`

with each configuration evaluated via a fixed downstream classifier (Logistic Regression) to isolate the effect of the embedding itself before introducing classifier choice as a second variable. The best-performing configuration was then carried forward for the full classifier comparison and for the unweighted-vs-TF-IDF-weighted aggregation comparison.

---

## Key Learning Objectives

Through this phase, I aim to:
* Understand how context-based embeddings differ from frequency-based representations, both conceptually and in preprocessing requirements.
* Empirically evaluate how embedding hyperparameters (context window, training epochs, training algorithm) affect downstream classification quality.
* Test the assumption that frequency-based reweighting (TF-IDF) improves embedding-based sentence vectors, and report the result honestly whether or not it confirms the initial hypothesis.
* Compare Word2Vec-based classical models against the BoW/TF-IDF baselines established earlier in the project.
* Build toward sequence-aware models (BiLSTM) that consume these same embeddings, as the next stage beyond static sentence-vector pooling.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)

# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [3]:
dataset = pd.read_csv('/kaggle/input/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis/MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.1 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
import re
import contractions

corpus = []
for i in range(0, len(dataset)):
    review = dataset['review'].iloc[i]
    # Fix contractions (don't -> do not)
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    # remove possessive 's before stripping other punctuation
    review = re.sub(r"'s\b", '', review)
    # Keep only letters and spaces
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split into tokens
    words = review.split()
    # NO stopword removal (keep context), NO lemmatization (keep word forms distinct)
    tokens = words
    corpus.append(tokens)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Encoding Categorical data Using Label Encoding**

In [10]:
from sklearn.preprocessing import LabelEncoder
y = dataset.iloc[:, 0].values
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [12]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.20, random_state = 0)

# **Creating the Word2Vec model**

In [14]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=X_train,
    vector_size=300,
    window=10,
    min_count=2,
    sg=1,          # skip-gram, per your established methodology
    epochs=20,
    seed=0,
    workers=1      # set to 1 for reproducibility; multi-worker training is non-deterministic even with a seed
)

## **Vocabulary Size Check:** Displays the total count of unique words stored in the Word2Vec model's dictionary (`model.wv`).

In [15]:
print("Vocabulary size:", len(model.wv))

Vocabulary size: 42288


## **Sentiment Similarity Test:** Iterates through sample sentiment words, checks for vocabulary presence, and prints the top 5 most similar words based on cosine distance using `model.wv.most_similar()`.

In [16]:
test_words = [
    'excellent',
    'wonderful',
    'fantastic',
    'terrible',
    'awful',
    'boring',
    'amazing'
]

for word in test_words:
    if word in model.wv:
        print(f"\nSimilar words to '{word}':")
        for similar_word, score in model.wv.most_similar(word, topn=5):
            print(f"  {similar_word}: {score:.4f}")


Similar words to 'excellent':
  superb: 0.5378
  outstanding: 0.5373
  great: 0.5306
  amazing: 0.5252
  eves: 0.5218

Similar words to 'wonderful':
  great: 0.6062
  excellent: 0.4982
  tink: 0.4808
  amazing: 0.4761
  magnificent: 0.4750

Similar words to 'fantastic':
  great: 0.5182
  brilliant: 0.4868
  excellent: 0.4767
  incredible: 0.4626
  wonderful: 0.4613

Similar words to 'terrible':
  bad: 0.6127
  awful: 0.6037
  horrible: 0.5528
  innane: 0.5456
  unbelieveable: 0.5279

Similar words to 'awful':
  terrible: 0.6037
  bad: 0.5606
  horrible: 0.5206
  missable: 0.5059
  hokie: 0.4934

Similar words to 'boring':
  unsubstantial: 0.5699
  dull: 0.5512
  tarintino: 0.5479
  pointless: 0.5306
  feedings: 0.5219

Similar words to 'amazing':
  excellent: 0.5252
  incredible: 0.5244
  wonderful: 0.4761
  awesome: 0.4672
  outstanding: 0.4656


## Mean Word Vector Aggregation (Dense Feature Extraction)

Translates variable-length token sequences into fixed-dimensional document embeddings for traditional machine learning models.

* **Vocabulary Lookup:** Filters input tokens against `model.wv` to retrieve corresponding Word2Vec embeddings.
* **Centroid Calculation:** Computes the element-wise mean (`axis=0`) over valid word vectors, producing a unigram centroid representation per text sample.
* **Zero-Vector Fallback:** Returns a 300-dimensional zero vector for samples containing exclusively out-of-vocabulary (OOV) terms.
* **Feature Matrix Generation:** Transforms train and test splits into 2D NumPy arrays of shape `(n_samples, 300)`.

In [17]:
def get_average_vector(tokens, model, vector_size):
    # Only keep words that are actually in the trained vocabulary
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    if len(valid_vectors) == 0:
        # Edge case: review has no words in vocab (rare, but handle it)
        return np.zeros(vector_size)
    
    return np.mean(valid_vectors, axis=0)

vector_size = 300
X_train_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_train])
X_test_w2v = np.array([get_average_vector(tokens, model, vector_size) for tokens in X_test])

print("X_train shape:", X_train_w2v.shape)

X_train shape: (19923, 300)


## TF-IDF Weighted Document Embeddings (No Data Leakage)

Combines semantic word embeddings with statistical term frequency weights to generate rich document-level feature representations.

* **Leakage-Free Fitting:** Fits `TfidfVectorizer` exclusively on `X_train` strings, extracting Inverse Document Frequency (IDF) weights into `tfidf_weights`.
* **Fallback Strategy:** Maps missing or single-character tokens (filtered by scikit-learn's default token regex) to `1.0`, matching the theoretical minimum IDF baseline ($\ln(1) + 1$).
* **Weighted Aggregation:** Computes `np.average(vectors, axis=0, weights=weights)` to prioritize high-information, domain-specific terms while down-weighting ubiquitous words.
* **Output Matrices:** Transforms train and test token lists into dense 2D feature matrices (`X_train_tfidf_w2v` and `X_test_tfidf_w2v`) of shape `(n_samples, vector_size)`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on raw string texts (train only, no leakage)
tfidf = TfidfVectorizer()
tfidf.fit([' '.join(tokens) for tokens in X_train])
tfidf_weights = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def get_tfidf_w2v_vector(tokens, model, vector_size):
    valid_tokens = [w for w in tokens if w in model.wv]
    if not valid_tokens:
        return np.zeros(vector_size)
    
    # sklearn's idf = ln((1+n)/(1+df)) + 1, whose theoretical minimum is exactly 1.0
    # (a word appearing in every document). TfidfVectorizer's default token_pattern
    # also drops single-char tokens (e.g. 'a', 'i'), so they won't be in tfidf_weights
    # even though they're valid Word2Vec tokens -> fall back to that minimum weight.
    weights = [tfidf_weights.get(w, 1.0) for w in valid_tokens]
    vectors = [model.wv[w] for w in valid_tokens]
    
    return np.average(vectors, axis=0, weights=weights)

vector_size = 100
X_train_tfidf_w2v = np.array([get_tfidf_w2v_vector(tokens, model, vector_size) for tokens in X_train])
X_test_tfidf_w2v  = np.array([get_tfidf_w2v_vector(tokens, model, vector_size) for tokens in X_test])

# **Training the XGBoost model on the Training set**

In [18]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(max_iter=1000, random_state = 0)
classifier.fit(X_train_w2v,y_train)

LogisticRegression(max_iter=1000, random_state=0)

# **Predicting the Test set results**

In [19]:
y_pred = classifier.predict(X_test_w2v)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


In [20]:
y_proba = classifier.predict_proba(X_test_w2v)[:, 1]
print(np.concatenate((y_proba.reshape(len(y_proba),1), y_test.reshape(len(y_test),1)),1))

[[0.80626403 1.        ]
 [0.68142625 1.        ]
 [0.12827415 0.        ]
 ...
 [0.12018716 0.        ]
 [0.07801205 0.        ]
 [0.24436482 0.        ]]


# **Evaluating the Model Performance**

In [21]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score


print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_proba))


accuracies = cross_val_score(estimator=classifier, X=X_train_w2v, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2199  321]
 [ 277 2184]]

Accuracy Score:
0.8799437863882754

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      2520
           1       0.87      0.89      0.88      2461

    accuracy                           0.88      4981
   macro avg       0.88      0.88      0.88      4981
weighted avg       0.88      0.88      0.88      4981


ROC-AUC Score:
0.9492784259850493

Mean Accuracy:
0.8790844752296341

Standard Deviation:
0.001329884184057786


# Word2Vec Performance Analysis for Text Classification (Logistic Regression)

# 1. Objective

The objective of this experiment was to evaluate **Word2Vec embeddings** (via Gensim) as a feature representation for binary text classification, using **Logistic Regression** as a fixed downstream classifier throughout so that every result isolates the effect of the embedding itself rather than the classifier.

Unlike the BoW/TF-IDF experiments, where the *classifier* was the main variable at a fixed vectorizer, this experiment fixes the classifier and varies the **embedding configuration**: training algorithm (skip-gram vs. CBOW), vector size, context window, training epochs, and — in a second phase — the pooling strategy used to turn a variable-length sequence of word vectors into a single fixed-length document vector.

The experiment aims to answer the following research questions:

* Does skip-gram (`sg=1`) meaningfully outperform CBOW (`sg=0`) for this task?
* How do vector size, window size, and training epochs each affect downstream accuracy?
* Does TF-IDF-weighting the word vectors during pooling improve on simple mean pooling?
* How does the best Word2Vec result compare to the best BoW and TF-IDF Logistic Regression results?

---

# 2. Experimental Setup

## Dataset

* Final test set: **4,981 documents** (2,520 negative / 2,461 positive).

## Embedding Pipeline

Word2Vec models were trained with Gensim (`min_count=2, seed=0, workers=1` fixed throughout for reproducibility). Two pooling strategies were tested:

**Phase 1 — Mean pooling**: each document's vector is the simple average of its words' embeddings (falling back to a zero vector if no words are in the trained vocabulary).

**Phase 2 — TF-IDF-weighted pooling**: each document's vector is a weighted average, using each word's TF-IDF idf weight (fit on the training set only, no leakage). The implementation includes a documented, deliberate fallback: sklearn's idf formula has a theoretical minimum of 1.0, and its default tokenizer drops single-character tokens (e.g. "a", "i") that can still be valid Word2Vec vocabulary — such words fall back to that minimum weight rather than being silently dropped or crashing. This is careful, well-reasoned edge-case handling worth noting.

Classifier: `LogisticRegression(max_iter=1000, random_state=0)` for every run, no hyperparameter search — this experiment is about the embedding, not the classifier.

---

# 3. Embedding Configurations Explored

| Phase | Run | vector_size | window | sg | epochs | Pooling |
| ----- | --- | ------------ | ------ | -- | ------ | -------- |
| 1 | 1 | 100 | 5 | 1 (skip-gram) | 5 (default) | Mean |
| 1 | 2 | 200 | 5 | 1 | 5 | Mean |
| 1 | 3 | 300 | 5 | 1 | 5 | Mean |
| 1 | 4 | 300 | 5 | **0 (CBOW)** | 5 | Mean |
| 1 | 5 | 300 | 7 | 1 | 5 | Mean |
| 1 | 6 | 300 | 10 | 1 | 5 | Mean |
| 1 | 7 | 300 | 10 | 1 | **10** | Mean |
| 1 | 8 | 300 | 10 | 1 | **20** | Mean |
| 2 | 9 | 100 | 5 | 1 | 5 | **TF-IDF weighted** |
| 2 | 10 | 300 | 10 | 1 | 20 | **TF-IDF weighted** |

No automated hyperparameter search (RandomizedSearchCV, HalvingRandomSearchCV, etc.) was used — every configuration here was manually specified and evaluated exactly as configured, so there is no search-vs-evaluated mismatch to check for in this experiment, unlike several of the earlier tree-based reports.

Two runs (5 and 6) included a qualitative sanity check — nearest neighbors of the word "great" — confirming the embedding space captures reasonable synonymy (wonderful, marvellous, fantastic, terrific). One run (7, `epochs=10`) surfaced an odd outlier in that neighbor list ("pidgin"), likely corpus-specific noise rather than a methodology problem, but worth a quick look if embedding quality becomes important later.

---

# 4. Experimental Results

| Run | Configuration | Accuracy | ROC-AUC | CV Mean | CV Std |
| --- | -------------- | -------- | ------- | ------- | ------ |
| 1 | 100d, window=5, skip-gram, mean pool | 86.15% | 0.9316 | 85.72% | 0.0012 |
| 2 | 200d, window=5, skip-gram, mean pool | 86.41% | 0.9359 | 86.15% | 0.0012 |
| 3 | 300d, window=5, skip-gram, mean pool | 86.59% | 0.9363 | 86.24% | 0.0009 |
| 4 | 300d, window=5, **CBOW**, mean pool | 83.82% | 0.9164 | 83.57% | 0.0022 |
| 5 | 300d, window=7, skip-gram, mean pool | 86.95% | 0.9400 | 86.82% | 0.0006 |
| 6 | 300d, window=10, skip-gram, mean pool | 87.49% | 0.9439 | 87.24% | 0.0017 |
| 7 | 300d, window=10, skip-gram, epochs=10, mean pool | 87.55% | 0.9477 | 87.64% | **0.0002** |
| **8** | **300d, window=10, skip-gram, epochs=20, mean pool** | **88.00%** | **0.9493** | 87.91% | 0.0013 |
| 9 | 100d, window=5, skip-gram, **TF-IDF pool** | 85.67% | 0.9276 | 85.73% | 0.0023 |
| 10 | 300d, window=10, skip-gram, epochs=20, **TF-IDF pool** | 87.07% | 0.9423 | 87.29% | 0.0017 |

---

# 5. Performance Analysis

## Effect of Vector Size (Runs 1-3, Isolated Comparison)

| Vector Size | Accuracy |
| ------------- | -------- |
| 100 | 86.15% |
| 200 | 86.41% |
| 300 | 86.59% |

A small, monotonic gain from more dimensions, with diminishing returns already visible (100→200 gained 0.26 points, 200→300 gained only 0.18).

## Effect of Skip-gram vs. CBOW (Runs 3 vs. 4, Isolated Comparison)

| Algorithm | Accuracy |
| ---------- | -------- |
| Skip-gram (`sg=1`) | **86.59%** |
| CBOW (`sg=0`) | 83.82% |

Skip-gram outperforms CBOW by **2.77 points** at an otherwise identical configuration — the largest single effect of any variable tested in this experiment, and a direct confirmation of why `sg=1` matters. This connects directly to the earlier caught bug where a missing `sg=1` parameter caused a silent fallback to CBOW — this result quantifies exactly how much accuracy that bug would have cost if it had gone unnoticed.

## Effect of Window Size (Runs 3, 5, 6, Isolated Comparison)

| Window | Accuracy |
| ------- | -------- |
| 5 | 86.59% |
| 7 | 86.95% |
| **10** | **87.49%** |

A consistent, monotonic gain from a wider context window, with no sign of plateauing by window=10 — a wider window may be worth testing further.

## Effect of Training Epochs (Runs 6, 7, 8, Isolated Comparison)

| Epochs | Accuracy |
| ------- | -------- |
| 5 (Gensim default) | 87.49% |
| 10 | 87.55% |
| **20** | **88.00%** |

More training epochs helped, with the biggest single jump (+0.45) coming from 10→20 rather than 5→10 — suggesting the default 5 epochs under-trains this embedding, and it may be worth testing even more epochs as a follow-up.

## Effect of Pooling Method (Runs 1 vs. 9, and 8 vs. 10 — Two Isolated Comparisons)

| Config | Mean Pooling | TF-IDF-Weighted Pooling | Change |
| ------ | -------------- | -------------------------- | ------ |
| 100d, window=5, epochs=5 | 86.15% | 85.67% | **-0.48** |
| 300d, window=10, epochs=20 (best config) | **88.00%** | 87.07% | **-0.93** |

TF-IDF-weighted pooling **underperformed simple mean pooling in both tested cases**, and the gap widened on the stronger base configuration rather than narrowing. A plausible explanation: TF-IDF weighting explicitly up-weights rare words, but with `min_count=2`, rare words in this corpus have the fewest training examples and therefore the least reliable embeddings — up-weighting exactly the vectors most likely to be noisy may be actively hurting the pooled representation, even though the same up-weighting strategy helps in the discrete BoW/TF-IDF representations (where it's the word's *identity*, not an embedding's *quality*, that's being weighted). This is a useful negative result: a technique that helps for sparse count-based features doesn't automatically transfer to dense embeddings.

---

# 6. Precision and Recall Analysis

### Skip-gram vs. CBOW (Runs 3 vs. 4)

| Run | Class | Precision | Recall | F1-score |
| --- | ----- | --------- | ------ | -------- |
| 3 (skip-gram) | 0 | 0.88 | 0.85 | 0.87 |
| 3 (skip-gram) | 1 | 0.85 | 0.88 | 0.87 |
| 4 (CBOW) | 0 | 0.85 | 0.83 | 0.84 |
| 4 (CBOW) | 1 | 0.83 | 0.85 | 0.84 |

### Best Model — Run 8 (Mean Pooling)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0 | 0.89 | 0.87 | 0.88 |
| 1 | 0.87 | 0.89 | 0.88 |

### Run 10 (TF-IDF-Weighted Pooling, Same Embedding as Run 8)

| Class | Precision | Recall | F1-score |
| ----- | --------- | ------ | -------- |
| 0 | 0.88 | 0.86 | 0.87 |
| 1 | 0.86 | 0.88 | 0.87 |

Class balance is consistently near-symmetric across every run in this experiment 

---

# 7. Cross-Validation Analysis

| Run | CV Mean | CV Std |
| --- | ------- | ------ |
| 6 (window=10, epochs=5) | 87.24% | 0.0017 |
| 7 (window=10, epochs=10) | 87.64% | **0.0002** |
| **8 (window=10, epochs=20)** | **87.91%** | 0.0013 |

Run 7's CV std (0.0002) is the tightest of any single result across this entire project so far — remarkably stable, even though its accuracy is slightly below Run 8's. Worth noting as a genuinely strong "runner-up" candidate if consistency is prioritized over the last half-point of accuracy.

---

# 8. Best Model Configuration

```python
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=X_train,
    vector_size=300,
    window=10,
    min_count=2,
    sg=1,
    epochs=20,
    seed=0,
    workers=1,
)

def get_average_vector(tokens, model, vector_size):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(valid_vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(valid_vectors, axis=0)

X_train_w2v = np.array([get_average_vector(tokens, model, 300) for tokens in X_train])
X_test_w2v = np.array([get_average_vector(tokens, model, 300) for tokens in X_test])

from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(max_iter=1000, random_state=0)
classifier.fit(X_train_w2v, y_train)
```

Performance:

* Accuracy = **88.00%**
* Precision (Class 0 / 1) = **0.89 / 0.87**
* Recall (Class 0 / 1) = **0.87 / 0.89**
* ROC-AUC = **0.9493**
* Cross-Validation Accuracy = **87.91%**
* Cross-Validation Standard Deviation = **0.0013**

---

# 9. Comparison with Logistic Regression on BoW and TF-IDF

| Representation | Best Config | Accuracy | ROC-AUC |
| ---------------- | ------------ | -------- | ------- |
| BoW | 30K features | 88.78% | 0.8880 |
| TF-IDF | 20K features | 89.46% | 0.8948 |
| **Word2Vec** | **300d, window=10, epochs=20, mean pool** | **88.00%** | **0.9493** |

This is a genuinely interesting three-way split: **Word2Vec has by far the highest ROC-AUC of the three, but the lowest accuracy.** Sparse BoW/TF-IDF representations — despite being conceptually simpler — retain sharper, more threshold-friendly signal for this classifier, while Word2Vec's mean-pooled vectors rank documents excellently but lose some of the crispness needed for confident hard classification at the default threshold. This is consistent with Section 6's finding and reinforces that mean pooling is a known weak point of word-vector-based document representations — exactly the gap that moving to a sequence model (RNN/LSTM, next on the roadmap) is designed to address, since it can learn a much richer way of combining word vectors than a simple average.

---

# 10. Discussion

Three findings define this experiment. First, skip-gram's ~2.8 point advantage over CBOW is the largest single effect found, and directly validates a fix already made earlier in this project. Second, TF-IDF-weighted pooling — despite sounding like a natural improvement — consistently underperformed simple mean pooling, a useful negative result worth remembering before assuming a technique that helps sparse representations will automatically help dense ones. Third, and most consequential for the roadmap: Word2Vec's unusually large ROC-AUC-to-accuracy gap suggests the embedding itself is high quality, but mean pooling is discarding some of that quality before it reaches the classifier. This strongly motivates the next roadmap step (RNN/LSTM), which replaces the simple average with a learned, order-aware way of combining word vectors.

Every individual hyperparameter tested (vector size, window, epochs) showed a small, consistent, monotonic improvement with no plateau reached — suggesting there's still headroom in the embedding configuration itself, separate from the pooling-method limitation.

---

# 11. Final Conclusion

This experiment evaluated Word2Vec embeddings across ten configurations, holding Logistic Regression fixed as the classifier throughout. The best model used a **300-dimensional skip-gram embedding, window=10, trained for 20 epochs, with simple mean pooling**, achieving **88.00% accuracy** and a notably strong **0.9493 ROC-AUC** — the highest ROC-AUC of any Logistic Regression result across BoW, TF-IDF, or Word2Vec, despite trailing both sparse representations on raw accuracy.

The most actionable open items are: proceed to the RNN/LSTM phase of the roadmap, which should directly address the mean-pooling limitation this experiment surfaced.